# Multi-Agent Financial News Analysis Pipeline

Financial analysts spend hours reading news, assessing sentiment, and synthesizing findings into actionable insights. In this notebook, we build an automated pipeline where multiple AI agents collaborate to do this work — fetching financial news, running sentiment analysis with FinGPT models, and producing a structured investment brief.

We use [AG2](https://ag2.ai) to orchestrate three specialized agents:
- A **News Researcher** that gathers and summarizes recent financial news for a given ticker
- A **Sentiment Analyst** that runs FinGPT sentiment classification on the headlines
- An **Investment Advisor** that synthesizes findings into an actionable brief

Each agent has access to real tools and works with live data from the Hugging Face Hub and financial APIs.

## Setup

In [ ]:
!pip install "ag2[openai]>=0.11.4,<1.0" huggingface_hub transformers torch yfinance -q

## Connecting to a Language Model

We power our agents with an open-source model via the Hugging Face Inference API, which provides an OpenAI-compatible endpoint. You'll need a [HF token](https://huggingface.co/settings/tokens).

In [ ]:
import os
from huggingface_hub import get_token
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager, LLMConfig

hf_token = get_token()

llm_config = LLMConfig({
    "model": "Qwen/Qwen2.5-Coder-32B-Instruct",
    "api_key": hf_token,
    "api_type": "openai",
    "base_url": "https://router.huggingface.co/v1",
})

## Building the Financial Analysis Tools

Before creating agents, we give them tools to work with real financial data:

1. **get_stock_info** — fetch recent price data and key metrics for a ticker using `yfinance`
2. **get_financial_news** — retrieve recent news headlines for a company via `yfinance`
3. **analyze_sentiment** — run FinGPT-style sentiment classification on financial text using a HuggingFace model

In [ ]:
import json
from typing import Annotated
from datetime import datetime, timedelta

import yfinance as yf


def get_stock_info(
    ticker: Annotated[str, "Stock ticker symbol, e.g. 'AAPL', 'MSFT', 'GOOGL'"],
) -> str:
    """Fetch recent stock price data and key financial metrics."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        # Get recent price history
        hist = stock.history(period="1mo")
        if hist.empty:
            return json.dumps({"error": f"No price data found for {ticker}"})

        current_price = hist["Close"].iloc[-1]
        month_ago_price = hist["Close"].iloc[0]
        price_change_pct = ((current_price - month_ago_price) / month_ago_price) * 100

        result = {
            "ticker": ticker,
            "company_name": info.get("longName", ticker),
            "sector": info.get("sector", "unknown"),
            "current_price": round(current_price, 2),
            "price_change_1mo_pct": round(price_change_pct, 2),
            "market_cap": info.get("marketCap", "N/A"),
            "pe_ratio": info.get("trailingPE", "N/A"),
            "52w_high": info.get("fiftyTwoWeekHigh", "N/A"),
            "52w_low": info.get("fiftyTwoWeekLow", "N/A"),
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


def get_financial_news(
    ticker: Annotated[str, "Stock ticker symbol"],
    max_headlines: Annotated[int, "Maximum number of headlines to return"] = 10,
) -> str:
    """Get recent financial news headlines for a stock ticker."""
    try:
        stock = yf.Ticker(ticker)
        news = stock.news

        if not news:
            return json.dumps({"ticker": ticker, "headlines": [], "message": "No recent news found"})

        headlines = []
        for item in news[:max_headlines]:
            content = item.get("content", item)
            provider = content.get("provider", {})
            pub_date = content.get("pubDate", "")
            headlines.append({
                "title": content.get("title", ""),
                "publisher": provider.get("displayName", ""),
                "published": pub_date[:10] if pub_date else "unknown",
            })

        return json.dumps({"ticker": ticker, "headlines": headlines}, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


def analyze_sentiment(
    texts: Annotated[list[str], "List of financial text snippets to analyze (headlines, summaries, etc.)"],
) -> str:
    """Run sentiment analysis on financial texts using a FinGPT-aligned model.

    Uses a financial sentiment model from HuggingFace to classify each text
    as positive, negative, or neutral with confidence scores.
    """
    from transformers import pipeline

    # Use a financial sentiment model — ProsusAI/finbert is well-established
    # and aligns with FinGPT's sentiment analysis focus
    try:
        classifier = pipeline(
            "sentiment-analysis",
            model="ProsusAI/finbert",
            return_all_scores=False,
        )

        results = []
        for text in texts:
            pred = classifier(text[:512])[0]  # Truncate to model max length
            results.append({
                "text": text[:100] + ("..." if len(text) > 100 else ""),
                "sentiment": pred["label"],
                "confidence": round(pred["score"], 4),
            })

        # Aggregate summary
        sentiments = [r["sentiment"] for r in results]
        summary = {
            "positive": sentiments.count("positive"),
            "negative": sentiments.count("negative"),
            "neutral": sentiments.count("neutral"),
            "total": len(results),
        }

        return json.dumps({"results": results, "summary": summary}, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})

## Creating the Agent Team

Each agent has a focused role and access to specific tools:

- **News Researcher** — fetches stock data and news, builds a factual overview
- **Sentiment Analyst** — runs FinGPT-style sentiment classification on the headlines
- **Investment Advisor** — synthesizes everything into a structured brief with a recommendation

We use AG2's decorator pattern to register which agent can call which tool.

In [ ]:
researcher = AssistantAgent(
    name="News_Researcher",
    system_message=(
        "You are a financial news researcher. Your job is to gather data about "
        "the requested stock ticker. Use get_stock_info to fetch price and metrics, "
        "then use get_financial_news to get recent headlines. "
        "Present your findings clearly: company overview, recent price action, "
        "and a numbered list of the most relevant headlines. "
        "Only use the tools provided — do not make up data."
    ),
    llm_config=llm_config,
)

analyst = AssistantAgent(
    name="Sentiment_Analyst",
    system_message=(
        "You are a financial sentiment analyst specializing in NLP-based analysis. "
        "When the News Researcher has gathered headlines, use analyze_sentiment "
        "to run sentiment classification on those headlines. "
        "Present results in a structured table showing each headline, its sentiment "
        "(positive/negative/neutral), and confidence score. "
        "Provide an overall sentiment summary. "
        "Only use the tools provided — do not invent scores."
    ),
    llm_config=llm_config,
)

advisor = AssistantAgent(
    name="Investment_Advisor",
    system_message=(
        "You are an investment advisor. Based on the News Researcher's data and "
        "the Sentiment Analyst's classification results, produce a structured "
        "investment brief. Include:\n"
        "1. Company snapshot (price, metrics, sector)\n"
        "2. News sentiment summary (% positive/negative/neutral)\n"
        "3. Key risks and catalysts identified from the news\n"
        "4. Overall outlook (bullish/bearish/neutral) with reasoning\n\n"
        "IMPORTANT: Add a disclaimer that this is AI-generated analysis, not "
        "financial advice. End your brief with TERMINATE."
    ),
    llm_config=llm_config,
)

executor = UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
)

# Register tools — Researcher and Analyst can call them, Executor runs them
for tool_fn, description in [
    (get_stock_info, "Fetch recent stock price data and key financial metrics for a ticker"),
    (get_financial_news, "Get recent financial news headlines for a stock ticker"),
    (analyze_sentiment, "Run sentiment analysis on a list of financial text snippets"),
]:
    executor.register_for_execution()(tool_fn)
    researcher.register_for_llm(description=description)(tool_fn)
    analyst.register_for_llm(description=description)(tool_fn)

## Running the Pipeline

Let's analyze a stock. The GroupChat manager coordinates the agents — the Researcher gathers data first, the Analyst runs sentiment classification, and the Advisor wraps up with a structured brief.

In [ ]:
group_chat = GroupChat(
    agents=[executor, researcher, analyst, advisor],
    messages=[],
    max_round=10,
    speaker_selection_method="auto",
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

executor.run(
    manager,
    message=(
        "Analyze NVDA (NVIDIA). Fetch the latest stock data and news headlines, "
        "run sentiment analysis on the headlines, and produce an investment brief "
        "with an overall outlook."
    ),
).process()

## Reviewing the Agent Conversation

Let's trace how the agents collaborated — who spoke, what tools they called, and how the analysis was built step by step:

In [ ]:
for msg in group_chat.messages:
    name = msg.get("name", msg.get("role", "unknown"))
    content = msg.get("content", "")
    if content and content.strip():
        print(f"{'='*60}")
        print(f">> {name}:")
        print(f"{content[:800]}")
        print()

## Try It Yourself

Change the ticker and question below to analyze any stock:

In [ ]:
# Change this to any ticker you want to analyze!
your_ticker = "TSLA"
your_question = (
    f"Analyze {your_ticker}. Fetch stock data and recent news, "
    f"run sentiment analysis on the headlines, and produce an "
    f"investment brief with an overall outlook and key risks."
)

group_chat_2 = GroupChat(
    agents=[executor, researcher, analyst, advisor],
    messages=[],
    max_round=10,
    speaker_selection_method="auto",
)

manager_2 = GroupChatManager(groupchat=group_chat_2, llm_config=llm_config)
executor.run(manager_2, message=your_question).process()

# Print the advisor's brief
for msg in reversed(group_chat_2.messages):
    if msg.get("name") == "Investment_Advisor" and msg.get("content", "").strip():
        print("Investment Brief:")
        print(msg["content"].replace("TERMINATE", "").strip())
        break

## What We Built

This notebook demonstrated a practical financial analysis pipeline where:

1. **The News Researcher** fetched real-time stock data and news via `yfinance`
2. **The Sentiment Analyst** ran FinGPT-style sentiment classification on headlines using FinBERT from HuggingFace
3. **The Investment Advisor** synthesized data + sentiment into a structured investment brief

Each agent has a focused role with access to real tools — they call live APIs, run actual ML inference, and work with real market data.

**Extending this pattern:**
- Swap FinBERT for a FinGPT fine-tuned model (e.g., `FinGPT/fingpt-sentiment_llama2-13b_lora`) for potentially better financial sentiment accuracy
- Add a **Risk Manager** agent that cross-references sentiment with volatility metrics
- Add an **SEC Filings** tool to pull 10-K/10-Q data for fundamental analysis
- Connect to the [FinGPT Forecaster](https://huggingface.co/FinGPT/fingpt-forecaster_dow30_llama2-7b_lora) for price movement predictions
- Run the pipeline on a portfolio of tickers for batch analysis

For more on multi-agent orchestration patterns, see the [AG2 documentation](https://docs.ag2.ai).